# Final POI recommendation baseline
Run with the project installed, or set `PYTHONPATH=src`. Binary labels stay unchanged; individual scores average the active preferences. GFAR aggregates the candidate union.


In [4]:
from ourney_travelsync import FEATURE_NAMES, load_places, AverageWithoutMisery
from ourney_travelsync.RecSys import calculate_preference_score, prepare_group, recommend_group

user = dict(zip(FEATURE_NAMES, [4, 5, 3, 2, 2, 3, 3, 3, 4, 1]))
example_poi = dict.fromkeys(FEATURE_NAMES, 0) | {"food": 1, "cafe": 1}
print(calculate_preference_score(user, example_poi))  # 4.5


4.5


In [5]:
places = load_places()
users = {
    "alice": user,
    "bob": dict(zip(FEATURE_NAMES, [3, 2, 5, 4, 3, 2, 4, 2, 5, 3])),
}
group = prepare_group(places, users, individual_top_n=100)
print("POIs:", len(places.place_ids), "Candidates:", len(group.poi_ids))
print("Shapes:", group.user_features.shape, group.poi_features.shape, group.individual_scores.shape)
metadata = {row["place_id"]: row for row in places.records}
for result in recommend_group(places, users, top_k=10):
    print(result.poi_id, metadata[result.poi_id]["place_name"], result.score)


POIs: 3591 Candidates: 200
Shapes: (2, 10) (200, 10) (2, 200)
tour:129156 가덕도 등대 0.1
tour:2841481 공극샌드커피 0.1
tour:2726843 가덕도 연대봉 0.08526315789473685
tour:2832700 구디너프 0.08526315789473685
tour:2782765 곰내연밭 0.07289750692520776
tour:2782628 그린노마드 0.07289750692520776
tour:127925 광안리해변 테마거리 0.06247068085726782
tour:2833257 그릿비 일광 0.06247068085726782
tour:2744597 구덕문화공원 0.05363436744653586
tour:2783360 꽃피는 4월 밀익는 5월 0.05363436744653586


In [6]:
awm_results = recommend_group(places, users, recommender=AverageWithoutMisery(2.5))
for result in awm_results:
    print(result.poi_id, metadata[result.poi_id]["place_name"], result.score)


tour:2875837 달맞이동산 4.5
tour:2875849 대천공원 4.5
tour:127149 민주공원 4.5
tour:126856 부산 중앙공원 4.5
tour:128108 스포원파크 4.5
tour:126108 해운대온천센터 4.5
kakao:1976509541 토르마린체험찜질방 4.5
kakao:25618927 허심청 호텔농심 4.5
kakao:8857943 녹천탕 4.5
kakao:23636732 대영해수온천 4.5


In [ ]:
gfar_results = recommend_group(places, users)
for result in gfar_results:
    print(result.poi_id, metadata[result.poi_id]["place_name"], result.score)


## AWM vs GFAR: shared satisfaction evaluation

Both algorithms use the same users, candidate pool (union of each member's top 100), and requested top-K. Evaluation uses the original 1-5 individual preference scores, not AWM's mean output or GFAR's marginal gain.

- **Mean satisfaction:** average of each member's mean score across the returned POIs.
- **Minimum member satisfaction:** lowest of those member averages, not the lowest single POI score.
- **Returned POIs:** reported because AWM may return fewer than K; empty lists have undefined satisfaction (N/A).

These metrics evaluate agreement with the current category-based scoring assumptions. Actual recommendation quality needs user feedback. A higher minimum member satisfaction means the least-satisfied member has a higher average predicted score; it does not guarantee every place is acceptable.


In [8]:
import numpy as np
from ourney_travelsync import GFAR

comparison_top_k = 10
comparison_candidate_top_n = 100
comparison_group = prepare_group(
    places, users, individual_top_n=comparison_candidate_top_n,
)
comparison_group.validate(require_scores=True)

# Generate both lists from exactly the same scored candidate pool.
comparison_results = {
    "AWM": AverageWithoutMisery(2.5).recommend(comparison_group, top_k=comparison_top_k),
    "GFAR": GFAR().recommend(comparison_group, top_k=comparison_top_k),
}


def evaluate_satisfaction(data, recommendations):
    """Evaluate returned POIs using the shared member-by-POI score matrix."""
    data.validate(require_scores=True)
    column_by_id = {poi_id: i for i, poi_id in enumerate(data.poi_ids)}
    selected_ids = [result.poi_id for result in recommendations]
    if len(set(selected_ids)) != len(selected_ids):
        raise ValueError("A recommendation list must contain unique POIs.")
    columns = [column_by_id[poi_id] for poi_id in selected_ids]
    if not columns:
        return {
            "returned_pois": 0,
            "mean_satisfaction": None,
            "minimum_member_satisfaction": None,
            "member_satisfaction": dict.fromkeys(data.user_ids, None),
        }
    selected_scores = data.individual_scores[:, columns]
    member_means = selected_scores.mean(axis=1)
    return {
        "returned_pois": len(columns),
        "mean_satisfaction": float(member_means.mean()),
        "minimum_member_satisfaction": float(member_means.min()),
        "member_satisfaction": dict(zip(data.user_ids, member_means.tolist())),
    }


comparison_metrics = {
    name: evaluate_satisfaction(comparison_group, results)
    for name, results in comparison_results.items()
}


def format_satisfaction(value):
    return "N/A" if value is None else f"{value:.3f}"


print(f"Shared candidates: {len(comparison_group.poi_ids)} | Requested top-K: {comparison_top_k}")
print("Satisfaction values use the same 1-5 preference scale. Higher is better.")
print(f"{'Algorithm':<12} {'Returned POIs':>14} {'Mean satisfaction':>20} {'Minimum member satisfaction':>30}")
for name, metrics in comparison_metrics.items():
    print(f"{name:<12} {metrics['returned_pois']:>14} "
          f"{format_satisfaction(metrics['mean_satisfaction']):>20} "
          f"{format_satisfaction(metrics['minimum_member_satisfaction']):>30}")

print("\nEach member's average score across the returned POIs:")
print(f"{'Member':<20} {'AWM':>10} {'GFAR':>10}")
for member in comparison_group.user_ids:
    awm_value = comparison_metrics['AWM']['member_satisfaction'][member]
    gfar_value = comparison_metrics['GFAR']['member_satisfaction'][member]
    print(f"{member:<20} {format_satisfaction(awm_value):>10} {format_satisfaction(gfar_value):>10}")


Shared candidates: 200 | Requested top-K: 10
Satisfaction values use the same 1-5 preference scale. Higher is better.
Algorithm     Returned POIs    Mean satisfaction    Minimum member satisfaction
AWM                      10                4.500                          4.000
GFAR                     10                3.875                          3.500

Each member's average score across the returned POIs:
Member                      AWM       GFAR
alice                     4.000      4.250
bob                       5.000      3.500
